# Module 4: Fix the broken models

**Project:** debug five broken training pipelines. Each one has a different, realistic bug. For each, you find the symptom, track down the root cause, fix it, and write a one-line note.

**Done when:**
- all five self-checks pass,
- you wrote a one-line **root cause** for each case,
- you wrote your own **debugging checklist** with at least 8 items.

**How to use this notebook**
1. Save your own copy (*File > Save a copy in Drive*). No GPU is needed: every case runs in seconds on the CPU.
2. Run the setup cells first. Then work through the five cases **in any order**.
3. Each case has a short story, a **broken** code cell, and a **Self-check** cell. Run the broken cell and look carefully at what it prints and plots. The bug is in the code. Find it, fix it, and re-run until the self-check passes.
4. Stuck? Open the hints one at a time (the first one is the smallest nudge). Only after your own attempt, open the solution page.
5. Fill in your root-cause notes and checklist at the end.

**A promise about the bugs:** each one is a mistake real people make. None of them raises an error. The code runs, and that is exactly why they are dangerous.

## Setup (given)

In [ ]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

def set_seed(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

# Three clusters in 10 dimensions. The centres are fixed, so every dataset below
# comes from the same underlying "world".
CENTERS = np.random.default_rng(123).normal(0, 2.0, size=(3, 10))

def make_blobs(n, spread=2.0, seed=0):
    rng = np.random.default_rng(seed)
    y = rng.integers(0, 3, n)
    X = CENTERS[y] + rng.normal(0, spread, size=(n, 10))
    return X.astype(np.float32), y.astype(np.int64)

def to_loader(X, y, batch_size=64, shuffle=True):
    return DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=batch_size, shuffle=shuffle)

def mlp(hidden=64):
    return nn.Sequential(nn.Linear(10, hidden), nn.ReLU(), nn.Linear(hidden, 3))

def predict_acc(model, X, y):
    """Correct accuracy helper (used by the cases)."""
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X)).argmax(1).numpy()
    return float((pred == y).mean())

print("Setup done.")

## How to debug a model that "runs but is wrong"

Most deep learning bugs do not crash. They quietly give bad results. Work through the pipeline **in this order**, and stop at the first thing that looks wrong:

1. **Data:** print a batch. Look at shapes, value ranges, labels, and their order. Are the splits clean?
2. **Labels and loss:** is the loss the right one for the task, and is it fed what it expects? What loss should a model that guesses get?
3. **Model:** do the output shapes make sense? Are train and evaluation modes used correctly?
4. **Optimizer:** is the learning rate sensible? Is the loop order right?

Two powerful tricks:
- **Overfit one batch.** A correct model and loop can drive the loss on a *single small batch* to nearly zero. If it cannot, something basic is broken. The helper below does this.
- **Compare with a baseline.** If you cannot beat a trivial baseline (always predict the most common class), something is wrong.

In [ ]:
def overfit_one_batch(model, loss_fn, optimizer, x, y, steps=100):
    """Train on ONE batch many times. A healthy model and loop reach a loss near 0."""
    model.train()
    for _ in range(steps):
        loss = loss_fn(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

x_demo, y_demo = next(iter(to_loader(*make_blobs(64, seed=9), batch_size=64)))
set_seed(0)
m = mlp()
final = overfit_one_batch(m, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=1e-2), x_demo, y_demo)
print(f"Healthy model, loss on one batch after 100 steps: {final:.4f}  (should be close to 0)")

---
## Case 1: "The test score is amazing... until it meets real data"

A teammate reports a **high test accuracy**. But when the model is tried on **fresh data** (samples it has truly never seen, like customers in the real world), it does much worse.

The pipeline below builds the dataset, standardizes the features, splits it, trains a model, and measures both scores. **Run it and compare `test_acc` with `fresh_acc`.**

*Keep the `return {...}` line as it is, because the self-check reads it.*

<details><summary>Hint 1</summary>Which of the two numbers is trustworthy? The test set should behave like fresh data. Why doesn't it?</details>
<details><summary>Hint 2</summary>Each original sample exists <b>5 times</b> in the data (copies with tiny noise). After a random split of rows, where can the copies of one sample end up?</details>
<details><summary>Hint 3</summary>Split the 600 <i>original</i> samples (the <code>groups</code>) into train and test first, then take the rows that belong to each. Also compute the mean and standard deviation from the <b>training rows only</b>.</details>

In [ ]:
def case1_pipeline(seed=0):
    set_seed(seed)
    # 600 original samples. Each one is copied 5 times with tiny noise ("augmentation"), giving 3000 rows.
    X0, y0 = make_blobs(600, spread=3.0, seed=1)
    noise = np.random.default_rng(5).normal(0, 0.05, size=(3000, 10)).astype(np.float32)
    X_raw = np.repeat(X0, 5, axis=0) + noise
    y = np.repeat(y0, 5)
    groups = np.repeat(np.arange(600), 5)        # which original sample each row came from

    # standardize the features
    mean, std = X_raw.mean(axis=0), X_raw.std(axis=0)
    X = (X_raw - mean) / std

    # split into train and test
    idx = np.random.permutation(len(X))
    cut = int(0.8 * len(X))
    train_idx, test_idx = idx[:cut], idx[cut:]

    model = mlp(128)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    loader = to_loader(X[train_idx], y[train_idx], 64)
    for epoch in range(60):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()

    test_acc = predict_acc(model, X[test_idx], y[test_idx])
    # fresh data the model has never seen, standardized the same way
    Xf, yf = make_blobs(2000, spread=3.0, seed=2)
    fresh_acc = predict_acc(model, (Xf - mean) / std, yf)
    return {"test_acc": test_acc, "fresh_acc": fresh_acc, "train_idx": train_idx, "test_idx": test_idx,
            "groups": groups, "mean": mean, "std": std, "X_raw": X_raw}

res1 = case1_pipeline()
print(f"test accuracy:  {res1['test_acc']:.3f}")
print(f"fresh accuracy: {res1['fresh_acc']:.3f}")

### Self-check: Case 1

In [ ]:
res1 = case1_pipeline()
leak = set(res1["groups"][res1["train_idx"]]) & set(res1["groups"][res1["test_idx"]])
assert not leak, f"{len(leak)} original samples have copies in BOTH the train and test sets, so the test score is not honest. Split so that each original sample lands on only one side."
_train_mean = res1["X_raw"][res1["train_idx"]].mean(axis=0)
assert np.allclose(res1["mean"], _train_mean, atol=1e-4), "The standardization statistics should be computed from the TRAINING rows only."
_gap = abs(res1["test_acc"] - res1["fresh_acc"])
assert _gap < 0.10, f"Test accuracy ({res1['test_acc']:.3f}) and fresh accuracy ({res1['fresh_acc']:.3f}) still differ by {_gap:.3f}. An honest test set should predict fresh-data performance."
print(f"Case 1 fixed. test {res1['test_acc']:.3f} vs fresh {res1['fresh_acc']:.3f}: the test score is now honest.")

---
## Case 2: "Accuracy looks fine, but the loss will not go down"

A model on an easy 3-class problem gets a decent **validation accuracy**. But the **training loss stalls around 0.6** and refuses to go lower. On data this clean, a healthy model should reach a much lower loss. Nobody knows why.

**Run the cell and look at the loss curve.**

<details><summary>Hint 1</summary>Accuracy only needs the <i>largest</i> output to be right. The loss cares about the actual values. Print one row of <code>model(x)</code>: what do the outputs look like?</details>
<details><summary>Hint 2</summary><code>nn.CrossEntropyLoss</code> already applies a softmax inside it, and expects raw scores (logits). What happens when the model also applies a softmax?</details>
<details><summary>Hint 3</summary>The model's last layer should be a plain <code>Linear</code>. Remove the extra layer.</details>

In [ ]:
def case2_pipeline(seed=0):
    set_seed(seed)
    X, y = make_blobs(3000, spread=2.0, seed=3)
    Xtr, ytr, Xva, yva = X[:2400], y[:2400], X[2400:], y[2400:]
    model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 3), nn.Softmax(dim=1))
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    loader = to_loader(Xtr, ytr, 64)
    losses = []
    for epoch in range(30):
        model.train(); total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(yb)
        losses.append(total / len(ytr))
    return {"model": model, "losses": losses, "val_acc": predict_acc(model, Xva, yva)}

res2 = case2_pipeline()
print(f"final training loss {res2['losses'][-1]:.3f}    validation accuracy {res2['val_acc']:.3f}")
plt.plot(res2["losses"]); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Case 2: training loss"); plt.show()

### Self-check: Case 2

In [ ]:
res2 = case2_pipeline()
assert res2["losses"][-1] < 0.35, f"The training loss is stuck at {res2['losses'][-1]:.2f}. On this easy data a healthy model gets well below 0.35. Check what the model outputs and what the loss expects."
assert res2["val_acc"] > 0.85, f"Validation accuracy is only {res2['val_acc']:.3f}."
print(f"Case 2 fixed. Final loss {res2['losses'][-1]:.3f}, validation accuracy {res2['val_acc']:.3f}.")

---
## Case 3: "The loss starts in the hundreds"

A model that should learn easily instead shows **huge loss values** and predicts almost at random.

First, a given helper: `lr_range_test` trains a fresh model for 3 epochs at each learning rate you give it and reports the final loss. Run the cell below, then look at the broken pipeline.

In [ ]:
def lr_range_test(lrs, epochs=3, seed=0):
    """Train a fresh model for a few epochs at each learning rate. Return {lr: final batch loss}."""
    X, y = make_blobs(3000, spread=2.0, seed=3)
    loader = to_loader(X[:2400], y[:2400], 64)
    out = {}
    for lr in lrs:
        set_seed(seed)
        model = mlp(64); opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
        for _ in range(epochs):
            for xb, yb in loader:
                loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
        out[lr] = round(float(loss.item()), 3)
    return out

print(lr_range_test([1e-5, 1e-3, 1e-1, 5.0]))

<details><summary>Hint 1</summary>Look at the actual loss numbers in the first epoch. A 3-class model that guesses has a loss near 1.1. What does a loss of several hundred tell you about the <i>size of the steps</i>?</details>
<details><summary>Hint 2</summary>Call <code>lr_range_test</code> with several learning rates, for example <code>[1e-4, 1e-3, 1e-2, 1e-1]</code>. Which values make the loss fall, and which blow it up?</details>
<details><summary>Hint 3</summary>For Adam, a learning rate between about 1e-3 and 1e-2 works well on this problem.</details>

In [ ]:
LR = 5.0      # the learning rate your teammate used

def case3_pipeline(lr, seed=0):
    set_seed(seed)
    X, y = make_blobs(3000, spread=2.0, seed=3)
    Xtr, ytr, Xva, yva = X[:2400], y[:2400], X[2400:], y[2400:]
    model = mlp(64)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = to_loader(Xtr, ytr, 64)
    losses = []
    for epoch in range(20):
        model.train(); total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(yb)
        losses.append(total / len(ytr))
    return {"model": model, "losses": losses, "val_acc": predict_acc(model, Xva, yva)}

res3 = case3_pipeline(LR)
print(f"loss: first epoch {res3['losses'][0]:.2f}, last epoch {res3['losses'][-1]:.2f}    validation accuracy {res3['val_acc']:.3f}")

### Self-check: Case 3

In [ ]:
res3 = case3_pipeline(LR)
_last = res3["losses"][-1]
assert np.isfinite(_last), "The loss became nan or infinite. The steps are far too large."
assert _last < 0.35, f"The final training loss is {_last:.2f}. A healthy run on this data ends below 0.35. Which setting controls the size of each step?"
assert res3["val_acc"] > 0.85, f"Validation accuracy is only {res3['val_acc']:.3f}."
print(f"Case 3 fixed with LR = {LR}. Final loss {_last:.3f}, validation accuracy {res3['val_acc']:.3f}.")

---
## Case 4: "It learned... the last thing it saw"

The training data was exported **sorted by class**. After training, the model's accuracy on a normal validation set is **poor**, and it seems to favour one class.

**Run the cell and look at the validation accuracy. Then print the first and last few training labels.**

<details><summary>Hint 1</summary>Print <code>y[:10]</code> and <code>y[-10:]</code> in the pipeline. What order are the training rows in?</details>
<details><summary>Hint 2</summary>Imagine one epoch: about 30 batches that are all class 0, then 30 of class 1, then 30 of class 2. Which class did the model see <i>most recently</i> when training ended, and what did the last updates teach it?</details>
<details><summary>Hint 3</summary>The <code>DataLoader</code> has a setting that controls the order examples are visited in.</details>

In [ ]:
def case4_pipeline(seed=0):
    set_seed(seed)
    X, y = make_blobs(3000, spread=2.0, seed=3)
    order = np.argsort(y, kind="stable")        # the file was saved sorted by class
    X, y = X[order], y[order]
    Xv, yv = make_blobs(1000, spread=2.0, seed=4)

    loader = to_loader(X, y, batch_size=32, shuffle=False)
    model = mlp(64)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    for epoch in range(3):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
    return {"model": model, "loader": loader, "val_acc": predict_acc(model, Xv, yv)}

res4 = case4_pipeline()
print(f"validation accuracy: {res4['val_acc']:.3f}")

### Self-check: Case 4

In [ ]:
res4 = case4_pipeline()
assert res4["val_acc"] > 0.85, f"Validation accuracy is only {res4['val_acc']:.3f}. Think about the order in which the model sees the training examples."
print(f"Case 4 fixed. Validation accuracy {res4['val_acc']:.3f}.")

---
## Case 5: "The score changes every time I run the evaluation"

A model with dropout and batch normalization scored about 90% when you checked it in your training script. But your separate **evaluation function reports about 35%**, and the number is slightly **different each time** you call it on the same model and data.

First run the given cell that trains the model. Then look at `evaluate5` and find the bug.

In [ ]:
def build_case5(seed=0):
    """Given and correct: trains a model with BatchNorm and Dropout, and builds a validation loader."""
    set_seed(seed)
    X, y = make_blobs(3000, spread=2.0, seed=3)
    model = nn.Sequential(nn.Linear(10, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 3))
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    loader = to_loader(X, y, 64)
    for epoch in range(15):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
    Xv, yv = make_blobs(1200, spread=2.0, seed=5)
    order = np.argsort(yv, kind="stable")                      # validation file is sorted by class
    return model, to_loader(Xv[order], yv[order], batch_size=100, shuffle=False)

print("build_case5 ready.")

<details><summary>Hint 1</summary>Call <code>evaluate5</code> twice on the same model. Are the answers identical? What does that tell you about the model's behaviour during evaluation?</details>
<details><summary>Hint 2</summary>Which layers behave differently during training and during evaluation? Also notice that each validation batch contains examples of only <i>one</i> class.</details>
<details><summary>Hint 3</summary>Switch the model to evaluation mode, and turn off gradient tracking, since evaluation does not need gradients.</details>

In [ ]:
model5, val_loader5 = build_case5()      # retrains a fresh model every time you run this cell

def evaluate5(model, loader):
    correct, n = 0, 0
    for xb, yb in loader:
        pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item()
        n += len(yb)
    return correct / n

acc_a = evaluate5(model5, val_loader5)
acc_b = evaluate5(model5, val_loader5)
print(f"first call: {acc_a:.3f}    second call: {acc_b:.3f}")

### Self-check: Case 5

In [ ]:
model5, val_loader5 = build_case5()
_a = evaluate5(model5, val_loader5)
_b = evaluate5(model5, val_loader5)
assert _a == _b, f"Two evaluations of the SAME model on the SAME data gave {_a:.3f} and {_b:.3f}. Evaluation must be deterministic. Which layers are random or batch-dependent in training mode?"
assert _a > 0.85, f"Accuracy is only {_a:.3f}. Some layers behave differently in training mode and evaluation mode."
assert not model5.training, "After evaluating, the model should be in evaluation mode."
print(f"Case 5 fixed. Evaluation is now deterministic: {_a:.3f} both times.")

---
## Your root causes and your debugging checklist

Write **one line for each case**: what the root cause was (not just what you changed). Then write your **own debugging checklist** of at least 8 items: the questions you will ask the next time a model misbehaves. Draw on what you learned in all five cases.

*Examples of the style:* "Overfit one batch first." and "Print the first and last labels of every split."


In [ ]:
root_causes = {
    1: "",   # YOUR CODE HERE: one line, for example "..."
    2: "",
    3: "",
    4: "",
    5: "",
}

checklist = [
    # YOUR CODE HERE: at least 8 items, each a short sentence
]

### Self-check: notes

In [ ]:
assert all(isinstance(v, str) and len(v.strip()) >= 15 for v in root_causes.values()) and len(root_causes) == 5, \
    "Write a root cause (at least a short sentence) for each of the 5 cases."
assert len(checklist) >= 8, f"Your checklist has {len(checklist)} items. Write at least 8."
assert all(len(item.strip()) >= 10 for item in checklist) and len(set(checklist)) == len(checklist), \
    "Each checklist item should be a real sentence, and no duplicates."
print("Notes check passed. Well done: you can now debug models that run but are wrong.")

## Stretch challenge (optional)

Create a **sixth broken pipeline** for a classmate: take a working pipeline from this notebook, plant one realistic bug that does not crash, and write a short story of the symptom (but not the cause). Examples: labels shuffled separately from the images, a learning rate that decays to zero too early, evaluating on the training set by mistake. Swap with a classmate and see if they can find it, with your hints.

*Your bug and its story:*

## Reflection and submission

Write three lines in your portfolio repo's `module-04/README.md`:
1. What you fixed
2. Which bug was hardest to find, and which clue finally revealed it
3. The top 3 items of your checklist

Then download this notebook (*File > Download > .ipynb*), push it to `module-04` in your portfolio repo, and submit the link in Moodle.